# 04. Cohort label table (case / control) (AD cohort)

One table mapping each embedding row to its case/control label.

`label == 1` means case (AD); `label == 0` means control (non_AD).

Outputs (in `output/embeddings/`):
- `cohort_labels.parquet`
- `cohort_labels.csv`


## 1. Load row -> cohort mapping

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_DIR = Path('output')
EMB_DIR = INPUT_DIR / 'embeddings'
EMB_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_MATRIX_PATH = INPUT_DIR / '07_feature_matrix.parquet'

ROW_ID = 'row_id'
PERSON_ID = 'person_id'
COHORT_GROUP_COL = 'cohort_group'
POSITIVE_COHORT_LABEL = 'AD'   # case = 1, control = 0

KEEP_PERSON_ID = True   # set False to drop person_id from the exported table

want = [c for c in [ROW_ID, PERSON_ID, COHORT_GROUP_COL]]
fm = pd.read_parquet(FEATURE_MATRIX_PATH, columns=want)
print('Loaded patient rows:', fm.shape)
print(fm[COHORT_GROUP_COL].value_counts())


## 2. Build and save the label table

In [ ]:
labels = fm.copy()
labels['label'] = (labels[COHORT_GROUP_COL] == POSITIVE_COHORT_LABEL).astype('int8')
labels['group_name'] = labels[COHORT_GROUP_COL].map(
    {POSITIVE_COHORT_LABEL: 'case'}).fillna('control')

cols = [ROW_ID]
if KEEP_PERSON_ID and PERSON_ID in labels.columns:
    cols.append(PERSON_ID)
cols += [COHORT_GROUP_COL, 'group_name', 'label']
labels = labels[cols]

LABELS_PARQUET = EMB_DIR / 'cohort_labels.parquet'
LABELS_CSV = EMB_DIR / 'cohort_labels.csv'
labels.to_parquet(LABELS_PARQUET, index=False)
labels.to_csv(LABELS_CSV, index=False)

print('label == 1 -> case (', POSITIVE_COHORT_LABEL, '); label == 0 -> control.')
print(labels['label'].value_counts())
print('Saved:', LABELS_PARQUET)
print('Saved:', LABELS_CSV)
labels.head()


## 3. Verify embeddings align to the label table

In [ ]:
for n in [10, 20, 30]:
    p = EMB_DIR / f'embedding_top{n}.parquet'
    if not p.exists():
        print(f'[top{n}] missing:', p)
        continue
    emb_ids = pd.read_parquet(p, columns=[ROW_ID])[ROW_ID].reset_index(drop=True)
    same = emb_ids.equals(labels[ROW_ID].reset_index(drop=True))
    print(f'[top{n}] rows={len(emb_ids)} | row_id aligned with labels: {same}')


## 4. Final manifest

In [ ]:
print('All deliverables in', EMB_DIR.resolve())
for p in sorted(EMB_DIR.glob('*')):
    if p.is_file():
        print(f'  {p.name:42s} {p.stat().st_size / 1024**2:8.2f} MB')
